In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel, AdamW
from datasets import load_dataset
import time
import matplotlib.pyplot as plt

# -------- CONFIG --------
MODEL = "bert-base-uncased"
MAX_LEN = 128
BATCH = 16
EPOCHS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- DATA --------
dataset = load_dataset("glue", "sst2")
tokenizer = BertTokenizer.from_pretrained(MODEL)

def tok(x):
    return tokenizer(x["sentence"], padding="max_length",
                     truncation=True, max_length=MAX_LEN)

dataset = dataset.map(tok, batched=True)
dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])

train_loader = DataLoader(dataset["train"], batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(dataset["validation"], batch_size=BATCH)

# -------- MODEL --------
class Model(nn.Module):
    def __init__(self, freeze=False, drop=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(MODEL)

        if freeze:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.fc = nn.Sequential(
            nn.Linear(768,256),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(256,1)
        )

    def forward(self, ids, mask):
        out = self.bert(input_ids=ids, attention_mask=mask)
        cls = out.last_hidden_state[:,0,:]
        return self.fc(cls)

# -------- TRAIN --------
def run(freeze=False, lr=2e-5, drop=0.3):
    model = Model(freeze, drop).to(DEVICE)
    opt = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    val_accs = []
    start = time.time()

    for e in range(EPOCHS):
        model.train()
        for b in train_loader:
            ids = b["input_ids"].to(DEVICE)
            mask = b["attention_mask"].to(DEVICE)
            y = b["label"].float().unsqueeze(1).to(DEVICE)

            opt.zero_grad()
            out = model(ids, mask)
            loss = loss_fn(out, y)
            loss.backward()
            opt.step()

        # eval
        model.eval()
        correct = 0
        with torch.no_grad():
            for b in val_loader:
                ids = b["input_ids"].to(DEVICE)
                mask = b["attention_mask"].to(DEVICE)
                y = b["label"].to(DEVICE)

                pred = (torch.sigmoid(model(ids,mask))>0.5).int().squeeze()
                correct += (pred==y).sum().item()

        acc = correct/len(val_loader.dataset)
        val_accs.append(acc)
        print(f"Epoch {e+1} Acc={acc:.4f}")

    print("Time:", time.time()-start)
    return val_accs

# -------- EXPERIMENTS --------
frozen = run(freeze=True)
full   = run(freeze=False)

lr1 = run(lr=2e-5)
lr2 = run(lr=5e-5)

drop1 = run(drop=0.1)
drop2 = run(drop=0.3)

# -------- PLOT --------
plt.plot(frozen,label="frozen")
plt.plot(full,label="full")
plt.legend(); plt.title("Freeze vs Full"); plt.show()

plt.plot(lr1,label="2e-5")
plt.plot(lr2,label="5e-5")
plt.legend(); plt.title("LR"); plt.show()

plt.plot(drop1,label="0.1")
plt.plot(drop2,label="0.3")
plt.legend(); plt.title("Dropout"); plt.show()

# -------- ATTENTION --------
def show_attention(text):
    m = Model().to(DEVICE)
    t = tokenizer(text, return_tensors="pt").to(DEVICE)
    out = m.bert(**t, output_attentions=True)
    att = out.attentions[-1][0][0].detach().cpu()

    plt.imshow(att)
    plt.title("Attention")
    plt.show()

show_attention("this movie was extremely good but slightly slow")